# 🏗️ Урок 14 — Декоратори як архітектурний інструмент

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Що таке цей урок?

У попередньому уроці (13) ми вже зустрічали декоратори: `@timer` вимірював час, а `logged` логував пайплайн. Ти бачив синтаксис — тепер ми розберемо **механізм і архітектурну логіку**.

Цей урок **не про синтаксис**. Він про системне мислення.

> 💡 **Головна ідея:**
> Декоратор = функція, яка **змінює поведінку** іншої функції, не чіпаючи її код.
> Замість того, щоб писати логування, авторизацію, кешування всередині кожної функції — ти виносиш це **назовні**.

---

## Чому це важливо?

Уяви: у тебе є 50 функцій. Кожна повинна:
- логувати свій виклик
- перевіряти авторизацію
- вимірювати час виконання

Що робить початківець? Копіює ці 3 блоки у кожну з 50 функцій.
Що робить архітектор? Пише **3 декоратори** і ставить їх над функціями.

| Без декораторів | З декораторами |
|---|---|
| Дублювання коду у кожній функції | Одне місце — одна відповідальність |
| Зміна логіки = правка 50 функцій | Зміна логіки = правка 1 декоратора |
| Складне тестування | Тестуємо декоратор окремо |

У архітектурі це називається **"Cross-cutting concerns"** — наскрізна функціональність, що не належить до бізнес-логіки, але потрібна скрізь.

---
## 🔄 Швидкий повтор з уроку 13

Урок 13 ми закінчили декоратором `@timer`. Згадай — він вимірював час:

```python
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"⏱ {func.__name__} виконалась за {end - start:.4f} сек")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))
```

**🔮 Передбачення:** У коді `@timer` — що відбувається в момент, коли Python читає рядок `@timer`? Чи виконується `slow_sum` одразу?

> Відповідь: ні. `@timer` — це скорочення для `slow_sum = timer(slow_sum)`. Python передає функцію у фабрику і **замінює ім'я** `slow_sum` на результат — тобто на об'єкт `wrapper`. Сама функція ще не викликалась.

---
## 1. Основний механізм: крок за кроком

Перш ніж писати декоратори з `@`, розберемо механізм **вручну** — без синтаксичного цукру.

In [ ]:
# Крок 1: Визначаємо декоратор
# Декоратор — це звичайна функція, що приймає функцію і повертає функцію
def my_decorator(func):

    # Крок 2: Визначаємо wrapper — нову функцію, яка огортає оригінал
    def wrapper():
        print("--- ДО ВИКЛИКУ ---")  # додаткова поведінка ПЕРЕД
        func()                        # викликаємо оригінальну функцію
        print("--- ПІСЛЯ ВИКЛИКУ ---") # додаткова поведінка ПІСЛЯ

    # Крок 3: Повертаємо wrapper як об'єкт (не викликаємо його!)
    return wrapper

# Визначаємо оригінальну функцію
def say_hi():
    print("Привіт!")

# Крок 4: Замінюємо оригінальну функцію — вручну, без @
say_hi = my_decorator(say_hi)

# Тепер say_hi — це wrapper, а не оригінальна функція!
say_hi()

**💬 Пояснення: 4 кроки механізму**

```
1. my_decorator(say_hi)  → отримує функцію як об'єкт
2. def wrapper()         → створює нову функцію в пам'яті
3. return wrapper        → повертає новий об'єкт (не результат!)
4. say_hi = ...          → замінює ім'я на нову функцію
```

Після кроку 4 ім'я `say_hi` **більше не вказує на оригінал**. Воно вказує на `wrapper`. Оригінальна функція жива — але досяжна лише через замикання всередині wrapper.

---
## 2. Синтаксичний цукор `@`

Python дає скорочення для `f = decorator(f)` — символ `@`. Це **те саме**, просто коротший запис.

In [ ]:
# Ось два записи — ідентичні за поведінкою:

# ─── Запис 1: без @ (явний) ───────────────────────────────────────────
def say_bye():
    print("Бувай!")

say_bye = my_decorator(say_bye)  # замінюємо вручну
say_bye()

print("---")

# ─── Запис 2: з @ (скорочений) ────────────────────────────────────────
@my_decorator
def say_thanks():
    print("Дякую!")

# say_thanks вже замінено на wrapper у момент читання @
say_thanks()

---
## 3. Wrapper з аргументами і поверненням значення

**🔮 Передбачення:** Що станеться, якщо оригінальна функція приймає аргументи, а wrapper — ні?

**🐛 Дебаг 1: Зламаний декоратор — знайди і виправ дві помилки**

```python
def uppercase_decorator(func):
    def wrapper():          # ← ПОМИЛКА 1: wrapper не приймає аргументи
        result = func()     # ← ПОМИЛКА 2: func викликається без аргументів
        return result.upper()
    return wrapper

@uppercase_decorator
def greet(name):
    return f"Hello, {name}"

print(greet("Alice"))  # TypeError: wrapper() takes 0 positional arguments but 1 was given
```

💡 **Підказка:** wrapper має бути **прозорим тунелем** — приймати будь-які аргументи і передавати їх далі. Використай `*args, **kwargs`.

In [ ]:
# Твоє виправлення:
def uppercase_decorator(func):
    def wrapper(???):
        result = func(???)
        return result.upper()
    return wrapper

@uppercase_decorator
def greet(name):
    return f"Hello, {name}"

print(greet("Alice"))  # HELLO, ALICE

**🐛 Дебаг 2: Декоратор повертає `None`**

```python
def timer_decorator(func):
    def wrapper(*args, **kwargs):
        print("Починаємо...")
        result = func(*args, **kwargs)
        return result
    # ← ПОМИЛКА: тут має бути return wrapper!

@timer_decorator
def slow_func():
    print("Працюю...")

slow_func()  # TypeError: 'NoneType' object is not callable
```

💡 **Пояснення:** Якщо `return wrapper` відсутній — функція `timer_decorator` повертає `None`. Тоді `slow_func = None`, і виклик `slow_func()` падає.

In [ ]:
# ✅ Правильний universal decorator — шаблон для всіх наступних
def uppercase_decorator(func):
    def wrapper(*args, **kwargs):       # *args — позиційні, **kwargs — іменовані
        result = func(*args, **kwargs)  # передаємо ВСЕ оригінальній функції
        return result.upper()           # модифікуємо результат
    return wrapper                      # ← ОБОВ'ЯЗКОВО! повертаємо функцію, не результат

@uppercase_decorator
def greet(name):
    return f"Hello, {name}"

print(greet("Alice"))         # HELLO, ALICE
print(greet(name="Bob"))      # HELLO, BOB  (kwargs теж працюють)

**💬 Мнемонічна схема: стек викликів у декораторі**

```
greet("Alice")
    ↓
wrapper("Alice")          ← викликається замість greet
    ↓
    func("Alice")         ← оригінальна функція всередині
    ↓
    result = "Hello, Alice"
    ↓
    return result.upper() ← модифікований вихід
    ↓
"HELLO, ALICE"
```

> 🏗️ **Архітектурний принцип:**
> Добре спроектований wrapper — це **прозорий тунель**: він перехоплює виклик, нічого не ламає і повертає результат оригіналу (можливо модифікований).

---
## 4. Патерн 1: Logger — Cross-cutting concerns

Уявімо систему: 10 функцій обробляють дані. Менеджер каже: "Логуйте кожен виклик!"

**Поганий підхід:** додати `print` у кожну функцію → 10 змін, дублювання, складне відключення.

**Правильний підхід:** написати один `@log_action` декоратор.

```mermaid
flowchart TD
    Call["📞 Виклик функції\ncalculate(5, 4)"] --> Logger["🔲 @log_action\nлогує назву + аргументи"]
    Logger --> Original["🎯 calculate(a, b)\nbізнес-логіка"]
    Original --> Result["📊 Результат"]
    Logger --> Log["📝 Лог: 'calculate(5, 4)'"]
    style Logger fill:#2c3e50,color:#fff
    style Original fill:#e74c3c,color:#fff
```

In [ ]:
# ===== Декоратор-логер =====

def log_action(func):
    """Логує назву функції та аргументи при кожному виклику."""
    def wrapper(*args, **kwargs):
        # func.__name__ — ім'я оригінальної функції
        print(f"📞 Виклик: {func.__name__}(args={args}, kwargs={kwargs})")
        result = func(*args, **kwargs)  # виконуємо оригінал
        print(f"✅ Результат: {result}")
        return result                   # повертаємо без змін
    return wrapper


# Застосовуємо до будь-якої функції — не чіпаємо її код
@log_action
def multiply(a, b):
    return a * b

@log_action
def greet(name, greeting="Привіт"):
    return f"{greeting}, {name}!"


multiply(5, 4)
print("---")
greet("Вікторе", greeting="Вітаю")

**🧩 Завдання 1 (легке): `@timer_log`**

Напиши декоратор `timer_log`, який логує **час виконання** функції у мілісекундах. Формат виводу:
```
⏱ [slow_sum] виконано за 23.45 мс
```

💡 **Підказка:** `time.time()` повертає секунди. Помнож на 1000 для мілісекунд.

In [ ]:
import time

# Твоє рішення:
def timer_log(func):
    def wrapper(*args, **kwargs):
        pass  # напиши реалізацію
    return wrapper

@timer_log
def slow_sum(n):
    return sum(range(n))

slow_sum(1_000_000)

---
## 5. Патерн 2: Authorization — контроль доступу

Декоратор може **зупинити виконання** функції, якщо умова не виконана. Це класичний патерн авторизації.

```mermaid
flowchart TD
    Request["📨 Запит на виконання"] --> Auth["🔲 @requires_admin\nперевіряє роль"]
    Auth -->|"role == admin ✅"| Func["🎯 Функція виконується"]
    Auth -->|"role != admin ❌"| Error["🚫 PermissionError"]
    style Auth fill:#2c3e50,color:#fff
    style Func fill:#27ae60,color:#fff
    style Error fill:#e74c3c,color:#fff
```

In [ ]:
# Глобальний контекст поточного користувача (в реальному застосунку — з токена або сесії)
USER_CONTEXT = {"role": "guest", "name": "Аноним"}


def requires_admin(func):
    """Декоратор: дозволяє виконання лише адміністраторам."""
    def wrapper(*args, **kwargs):
        role = USER_CONTEXT.get("role")       # читаємо поточну роль
        if role != "admin":                    # перевірка доступу
            raise PermissionError(
                f"🚫 Доступ заборонено. Роль '{role}' не має прав адміністратора."
            )
        return func(*args, **kwargs)           # якщо все ок — виконуємо
    return wrapper


@requires_admin
def delete_all_data():
    print("🗑️ Дані видалено!")

@requires_admin
def view_reports():
    print("📊 Звіти доступні.")


# Тест 1: гість намагається отримати доступ
try:
    delete_all_data()
except PermissionError as e:
    print(e)

print("---")

# Тест 2: змінюємо роль на адміна
USER_CONTEXT["role"] = "admin"
delete_all_data()  # тепер проходить
view_reports()

**🧩 Завдання 2 (середнє): `@requires_role(role)`**

Узагальни `requires_admin`: напиши **фабрику декораторів** `requires_role(role)`, яка приймає будь-яку роль. Тоді замість `@requires_admin` можна писати `@requires_role("admin")` або `@requires_role("moderator")`.

💡 **Підказка:** Потрібен третій рівень вкладеності: `requires_role(role)` → `real_decorator(func)` → `wrapper(*args, **kwargs)`

In [ ]:
# Твоє рішення:
def requires_role(role):
    def real_decorator(func):
        def wrapper(*args, **kwargs):
            pass  # реалізуй перевірку
        return wrapper
    return real_decorator

# Тести:
# @requires_role("admin")
# def admin_panel(): print("Admin panel")
#
# @requires_role("moderator")
# def moderate_content(): print("Moderating...")

---
## 6. Патерн 3: Middleware Chain — стек декораторів

Декоратори можна **стекувати** — як шари цибулини або middleware в веб-фреймворках.

**🔮 Передбачення:** У якому порядку застосуються теги, якщо написати `@bold` над `@italic`?

```mermaid
flowchart TD
    Input["📥 Input\n'Python'"] --> Bold["🔲 @bold\nобгортає в bold"]
    Bold --> Italic["🔲 @italic\nобгортає в italic"]
    Italic --> Core["🎯 format_text()\nповертає текст"]
    Core --> Italic2["↩ italic wrapper\nдодає теги"]
    Italic2 --> Bold2["↩ bold wrapper\nдодає теги"]
    Bold2 --> Output["📤 Output\n'<b><i>Python</i></b>'"]
    style Bold fill:#2c3e50,color:#fff
    style Italic fill:#34495e,color:#fff
    style Core fill:#e74c3c,color:#fff
```

In [ ]:
def bold(func):
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

def underline(func):
    def wrapper(*args, **kwargs):
        return f"<u>{func(*args, **kwargs)}</u>"
    return wrapper


# Декоратори застосовуються знизу вгору:
# f = bold(italic(format_text))
# тобто спочатку italic, потім bold огортає italic
@bold
@italic
def format_text(text):
    return text

print(format_text("Python"))  # <b><i>Python</i></b>

print("---")

# Три шари
@bold
@italic
@underline
def format_triple(text):
    return text

print(format_triple("Middleware"))  # <b><i><u>Middleware</u></i></b>

**💬 Пояснення: порядок стекування**

```python
@bold
@italic
def f(): ...

# Еквівалентно:
f = bold(italic(f))
```

Читання відбувається **знизу вгору**: спочатку `italic` огортає `f`, потім `bold` огортає результат.
Виконання відбувається **ззовні всередину**: спочатку `bold.wrapper`, потім `italic.wrapper`, потім оригінальна `f`.

Саме так працює **middleware** у FastAPI, Django, Express:

```
Запит → CORS → Auth → RateLimit → Validation → Handler → Response
```

---
## 7. Фабрики декораторів — декоратори з аргументами

Іноді потрібно передати параметр у декоратор: `@repeat(times=3)` або `@censor(words=["spam"])`.
Це неможливо зробити з простим декоратором. Потрібна **фабрика декораторів** — ще один рівень.

**🔮 Передбачення:** Що саме повертає `repeat(times=3)`? Це декоратор чи wrapper?

In [ ]:
# ─── Фабрика декораторів (3 рівні вкладеності) ─────────────────────────
#
# repeat(times=3)   ← рівень 1: фабрика, захоплює 'times'
#   real_decorator(func)  ← рівень 2: справжній декоратор, захоплює 'func'
#     wrapper(*args)      ← рівень 3: виконується при кожному виклику

def repeat(times):
    """Фабрика: повертає декоратор, що повторює виклик N разів."""
    def real_decorator(func):
        def wrapper(*args, **kwargs):
            last_result = None
            for i in range(times):          # times захоплено із зовнішнього scope
                last_result = func(*args, **kwargs)
            return last_result              # повертаємо результат останнього виклику
        return wrapper
    return real_decorator                   # повертаємо справжній декоратор


@repeat(times=3)
def ring_bell():
    print("🔔 Дзінь!")

@repeat(times=2)
def greet(name):
    print(f"Привіт, {name}!")

ring_bell()
print("---")
greet("Вікторе")

**💬 Пояснення: замикання у трирівневій структурі**

```
@repeat(times=3)
def ring_bell(): ...

# Розгортається ось так:
decorator = repeat(3)        # отримуємо real_decorator (times=3 захоплено)
ring_bell = decorator(ring_bell)  # отримуємо wrapper (func захоплено)
```

Коли ти пишеш `@repeat(times=3)`, Python:
1. Викликає `repeat(3)` → отримує `real_decorator` (з `times=3` у замиканні)
2. Передає `ring_bell` у `real_decorator` → отримує `wrapper` (з `func` і `times` у замиканні)
3. Замінює ім'я `ring_bell` на `wrapper`

**🧩 Завдання 3 (важке): `@censor(banned_words)`**

Напиши фабрику декораторів `censor(banned_words)`. Вона повинна:
1. Приймати список заборонених слів
2. Перед викликом оригінальної функції замінювати заборонені слова у першому аргументі на `"***"`
3. Передавати очищений текст у функцію

```python
@censor(banned_words=["spam", "bad"])
def publish_post(text):
    print(f"Публікуємо: {text}")

publish_post("This is bad spam content")
# Публікуємо: This is *** *** content
```

💡 **Підказка:** `str.replace(word, "***")` у циклі по `banned_words`.

In [ ]:
# Твоє рішення:
def censor(banned_words):
    def real_decorator(func):
        def wrapper(text, *args, **kwargs):
            pass  # реалізуй заміну слів
        return wrapper
    return real_decorator

# @censor(banned_words=["spam", "bad"])
# def publish_post(text):
#     print(f"Публікуємо: {text}")

---
## 8. `functools.wraps` — збереження метаданих

Є одна проблема з декораторами: після огортання функція **втрачає своє ім'я і документацію**.

**🔮 Передбачення:** Що виведе `complex_math.__name__` після застосування декоратора?

In [ ]:
# ❌ БЕЗ functools.wraps — метадані втрачені

def simple_decorator(func):
    def wrapper(*args, **kwargs):
        """Це рядок документації WRAPPER, не оригінальної функції."""
        return func(*args, **kwargs)
    return wrapper

@simple_decorator
def complex_math(x, y):
    """Виконує складну математику."""
    return x ** y

print(complex_math.__name__)  # 'wrapper' ← НЕПРАВИЛЬНО! Мало бути 'complex_math'
print(complex_math.__doc__)   # 'Це рядок документації WRAPPER...' ← неправильно!

# Це ламає: дебагінг, sphinx-документацію, pytest, FastAPI auto-docs

In [ ]:
import functools

# ✅ З functools.wraps — метадані збережені

def proper_decorator(func):
    @functools.wraps(func)       # копіює __name__, __doc__, __module__... з func у wrapper
    def wrapper(*args, **kwargs):
        """Цей docstring буде перезаписаний декоратором wraps."""
        return func(*args, **kwargs)
    return wrapper

@proper_decorator
def complex_math(x, y):
    """Виконує складну математику."""
    return x ** y

print(complex_math.__name__)  # 'complex_math' ✅
print(complex_math.__doc__)   # 'Виконує складну математику.' ✅

# ─── ПРАВИЛО: завжди додавай @functools.wraps(func) у production декораторах ───

---
## 9. `functools.lru_cache` — кешування як декоратор

Стандартна бібліотека Python містить готові декоратори. `@lru_cache` кешує результати функції — це мемоізація без жодного додаткового коду.

**🔮 Передбачення:** Без кешу `fibonacci(35)` робить мільйони рекурсивних викликів. Що станеться з `@lru_cache`?

In [ ]:
import functools
import time

# ─── Без кешу ─────────────────────────────────────────────────────────
def fib_slow(n):
    if n < 2:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

start = time.time()
print(fib_slow(35))
print(f"⏱ Без кешу: {(time.time() - start) * 1000:.1f} мс")

print("---")

# ─── З @lru_cache ─────────────────────────────────────────────────────
@functools.lru_cache(maxsize=None)  # maxsize=None → необмежений кеш
def fib_fast(n):
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)

start = time.time()
print(fib_fast(35))
print(f"⏱ З кешом: {(time.time() - start) * 1000:.3f} мс")

# Статистика кешу
print(fib_fast.cache_info())

**💬 Пояснення: як `@lru_cache` перехоплює виклик**

```
fib_fast(10)
    ↓
lru_cache.wrapper(10)
    ↓
    Чи є (10,) у кеші? → НІ → обчислити → зберегти → повернути
    Чи є (10,) у кеші? → ТАК → повернути миттєво
```

> LRU (Least Recently Used) — коли кеш переповнений, видаляється найстаріший рідко використовуваний запис.

**Важливо:** `@lru_cache` працює лише з **чистими функціями** (однаковий вхід → однаковий вихід). Якщо функція має побічні ефекти — кешування дасть неправильні результати.

---
## 10. 🚀 Міні-проект: Система обробки платежів

Збираємо все разом. Будуємо пайплайн для обробки платежу через стек декораторів.

**Архітектура:**

```mermaid
flowchart TD
    Call["process_payment(amount)"] --> Log["🔲 @log_action\nлогує виклик"]
    Log --> Auth["🔲 @requires_admin\nперевіряє роль"]
    Auth --> Valid["🔲 @validate_type(float)\nперевіряє тип"]
    Valid --> Core["🎯 process_payment\nбізнес-логіка"]
    style Log fill:#2c3e50,color:#fff
    style Auth fill:#8e44ad,color:#fff
    style Valid fill:#2980b9,color:#fff
    style Core fill:#e74c3c,color:#fff
```

Кожен шар відповідає за **одну** річ. Бізнес-логіка (`process_payment`) — чиста, без захаращення.

In [ ]:
import functools

# ─── Контекст ─────────────────────────────────────────────────────────
USER_CONTEXT = {"role": "admin", "name": "Viktor"}


# ─── Декоратор 1: Логування ────────────────────────────────────────────
def log_action(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"📞 [{func.__name__}] args={args} kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"✅ [{func.__name__}] → {result}")
        return result
    return wrapper


# ─── Декоратор 2: Авторизація ──────────────────────────────────────────
def requires_admin(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if USER_CONTEXT.get("role") != "admin":
            raise PermissionError(f"🚫 Доступ заборонено для ролі '{USER_CONTEXT['role']}'")
        return func(*args, **kwargs)
    return wrapper


# ─── Декоратор 3: Валідація типу ───────────────────────────────────────
def validate_type(expected_type):
    def real_decorator(func):
        @functools.wraps(func)
        def wrapper(value, *args, **kwargs):
            if not isinstance(value, expected_type):
                raise TypeError(
                    f"❌ Очікується {expected_type.__name__}, "
                    f"отримано {type(value).__name__}: {value!r}"
                )
            return func(value, *args, **kwargs)
        return wrapper
    return real_decorator


# ─── Бізнес-логіка: ЧИСТА, без захаращення ────────────────────────────
@log_action
@requires_admin
@validate_type(float)
def process_payment(amount):
    """Обробляє платіж у системі."""
    # Тут лише бізнес-логіка — без логування, авторизації, валідації
    commission = round(amount * 0.02, 2)
    net = round(amount - commission, 2)
    return {"status": "ok", "amount": amount, "commission": commission, "net": net}


# ─── Тести ─────────────────────────────────────────────────────────────
print("=== Тест 1: коректний платіж ===")
process_payment(100.50)

print("\n=== Тест 2: неправильний тип ===")
try:
    process_payment("100")  # рядок замість float
except TypeError as e:
    print(e)

print("\n=== Тест 3: недостатньо прав ===")
USER_CONTEXT["role"] = "guest"
try:
    process_payment(50.0)
except PermissionError as e:
    print(e)

---
## 11. Декоратори в реальних фреймворках

Тепер ти розумієш механізм — і можеш читати код FastAPI, pytest, SQLAlchemy **без магії**.

In [ ]:
# ─── Приклад 1: FastAPI — @app.get() ──────────────────────────────────
# Що відбувається насправді:
#
# @app.get("/users/{id}")   ← це фабрика декораторів!
# def get_user(id: int):    ← функція реєструється як обробник маршруту
#     return {"id": id}
#
# Еквівалентно:
# get_user = app.get("/users/{id}")(get_user)
#
# Декоратор: перехоплює HTTP-запит, парсить URL, викликає get_user,
# серіалізує результат у JSON-відповідь.
# Бізнес-логіка (get_user) — чиста.

print("FastAPI pattern — see above comment")

print("---")

# ─── Приклад 2: @dataclass ────────────────────────────────────────────
# @dataclass — КЛАСОВИЙ декоратор (приймає клас, повертає клас)
from dataclasses import dataclass

@dataclass
class Payment:
    amount: float
    currency: str = "UAH"
    status: str = "pending"

# @dataclass автоматично генерує __init__, __repr__, __eq__
p = Payment(amount=100.50)
print(p)          # Payment(amount=100.5, currency='UAH', status='pending')
print(repr(p))    # те саме — автоматично

print("---")

# ─── Приклад 3: @property ─────────────────────────────────────────────
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property              # перетворює метод на атрибут (getter)
    def area(self):
        """Площа кола — обчислюється, а не зберігається."""
        import math
        return math.pi * self._radius ** 2

c = Circle(5)
print(f"Площа: {c.area:.2f}")  # доступ як до атрибута, а не метода!

**Pytest і класові декоратори:**

```python
# @pytest.mark.skip — декоратор, що додає метадані до функції-тесту
@pytest.mark.skip(reason="Ще не реалізовано")
def test_payment_refund():
    assert refund(100) == 100

# @pytest.mark.parametrize — фабрика декораторів, генерує кілька тест-кейсів
@pytest.mark.parametrize("amount,expected", [
    (100.0, 98.0),
    (50.0,  49.0),
])
def test_net_amount(amount, expected):
    assert process_payment(amount)["net"] == expected
```

Тепер ти **розумієш** ці конструкції — вони не магія, а прості декоратори.

---
## ✅ Підсумок уроку

### 5 архітектурних принципів:

| # | Принцип | Що це означає на практиці |
|---|---|---|
| 1 | **Функції — дані** | Декоратор використовує те, що функцію можна передати і повернути |
| 2 | **Wrap, не mutate** | Ти огортаєш функцію в wrapper, не змінюєш її код |
| 3 | **`wrapper → func → result`** | Кожен wrapper — прозорий тунель з `*args, **kwargs` і `return result` |
| 4 | **Middleware через стек** | `@log @auth @validate` — це пайплайн, де порядок важливий |
| 5 | **Cross-cutting concerns** | Логування, безпека, кешування — поза бізнес-логікою, у декораторах |

---

### `@functools.wraps` — завжди!

```python
# Шаблон production-декоратора:
import functools

def my_decorator(func):
    @functools.wraps(func)          # ← завжди!
    def wrapper(*args, **kwargs):   # ← завжди *args, **kwargs
        # ... before ...
        result = func(*args, **kwargs)
        # ... after ...
        return result               # ← завжди повертай result
    return wrapper                  # ← повертай wrapper, не wrapper()
```

---

### Декоратори в реальних системах:

| Фреймворк | Декоратор | Що робить |
|---|---|---|
| FastAPI | `@app.get("/")` | Реєстрація route handler |
| pytest | `@pytest.mark.skip` | Метадані для test runner |
| Python stdlib | `@dataclass` | Генерація `__init__`, `__repr__` |
| Python stdlib | `@property` | Getter/setter для атрибутів |
| Python stdlib | `@functools.lru_cache` | Автоматичне мемоізаційне кешування |
| SQLAlchemy | `@validates` | Хук валідації при зміні атрибута |

---

### 3 концептуальних запитання:

1. Чим `@decorator` відрізняється від `decorator(func)` — є різниця у поведінці?
2. Навіщо `@functools.wraps(func)` — що станеться без нього в FastAPI?
3. Як порядок стекування `@log @auth @validate` впливає на потік виконання?

---

*Viktor Nikoriak · Module 2 · Lesson 14 · Decorators as an Architectural Pattern*